# 2장 — 컴퓨터의 손가락: 비트와 이진법 🔢

웹 챕터의 스위치 놀이터를 이번엔 **코드로** 돌려 봅니다 — 이진수 조립, 2ⁿ 천장,
그리고 같은 비트가 약속에 따라 255도 되고 −1도 되는 마법(2의 보수)까지.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter` — 위에서 아래로.

**이 노트북의 목표**: 칸의 재료(비트)로 수를 세는 법 —
**"n비트 칸에는 2ⁿ 가지 천장이 있다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch2')

## 1. 스위치로 수 조립하기 — 자리값의 합

42 = 32 + 8 + 2. 파이썬의 `bin()` 과 `int(문자열, 2)` 로 웹 실험 ①을 재현합니다.

In [ ]:
n = 42
print(f'{n} 을 이진수로 = {n:08b}   (자리값: 128 64 32 16 8 4 2 1)')

# 자리값 분해 — 켜진 스위치(1)의 자리값만 골라 더한다
places = [128, 64, 32, 16, 8, 4, 2, 1]
on = [p for p, b in zip(places, f'{n:08b}') if b == '1']
print(f'켜진 자리값: {on} → 합 = {sum(on)}')

assert int('00101010', 2) == 42, '이진수 00101010 을 다시 십진수로 = 42'
assert f'{42:08b}' == '00101010', '42 를 8비트 이진수로 = 00101010'
assert sum(on) == 42, '켜진 자리값의 합이 곧 그 수'
print()
print('✔ 확인: 어떤 수든 자리값(2의 거듭제곱) 몇 개의 합 — 십진법의 425 = 400+20+5 와 같은 원리')

## 2. 천장 계산 — 스위치 n개면 몇 가지?

스위치 하나 늘 때마다 × 2. 그리고 실무에서 바로 쓰는 계산 하나 —
**어휘 32,000개의 토큰 번호는 몇 비트면 담길까?**

In [ ]:
import math

print(' 비트 수 n   가짓수 2^n         비고')
for n, note in [(1, '예/아니오'), (8, '한 바이트 — int8 눈금 개수'), (15, '토큰 번호(어휘 32,000) 크기'),
                (16, '유니코드 기본 평면'), (32, 'IPv4 주소 전체')]:
    print(f'   {n:>4}     {2**n:>13,}   {note}')

assert 2**8 == 256 and 2**8 - 1 == 255, '8비트: 256가지, 부호 없는 최대 255'
assert 2**14 < 32000 <= 2**15, '토큰 번호 0~31,999 는 14비트론 모자라고 15비트면 충분'
assert math.ceil(math.log2(32000)) == 15, '같은 계산을 log₂ 로 — 2부 7장의 로그가 여기서 일한다'
print()
print('✔ 확인: n비트 칸에는 2^n 가지 천장이 있다 — LLM 입력의 첫 관문(토큰 번호)부터 이 계산')

## 3. 같은 비트, 다른 약속 — 2의 보수

`view()` 는 **비트는 그대로 두고 읽는 약속만 바꿉니다** — 부호 없는 8비트(0~255)를
int8(−128~127)로 다시 읽어 보죠. 그리고 시계 산수로 "왜 이 약속이 통하는지"까지.

In [ ]:
import numpy as np

bits = np.array([255, 128, 127, 254], dtype=np.uint8)   # 부호 없음으로 읽은 값들
as_int8 = bits.view(np.int8)                              # 같은 비트를 int8 약속으로 재독
for u, s in zip(bits, as_int8):
    print(f'비트 {u:08b} → 부호 없음 {u:>3}  /  int8 {s:>4}')

assert list(as_int8) == [-1, -128, 127, -2], '11111111=−1, 10000000=−128, 01111111=127'
assert np.iinfo(np.int8).min == -128 and np.iinfo(np.int8).max == 127, 'int8 범위 = −128~127'

# 시계 산수 — 256칸 시계에서 "5 빼기"는 "251 더하기"와 같다 (3 − 5 = −2 ≡ 254)
assert (3 - 5) % 256 == 254 and np.array([254], np.uint8).view(np.int8)[0] == -2
# 그래서 int8 의 127 + 1 은 −128 로 한 바퀴 감긴다 (wraparound)
wrap = (np.array([127], dtype=np.int8) + np.array([1], dtype=np.int8))[0]
print(f'\nint8 에서 127 + 1 = {wrap}  ← 시계가 한 바퀴!')
assert wrap == -128
print()
print('✔ 확인: 비트는 그대로, 약속만 다르다 — 그리고 그 약속(2의 보수) 덕에 덧셈 회로 하나로 음수까지')

## 4. 보너스 — 천장의 성장 곡선

비트 수에 따라 가짓수가 어떻게 자라는지 — 세로축이 로그인데도 직선이 됩니다
(5부 1장에서 본 "log₂ 가 지수의 역"이라는 그 관계).

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ns = np.arange(1, 33)
plt.figure(figsize=(6.5, 3.8))
plt.semilogy(ns, 2.0 ** ns, 'o-', color='tab:purple', markersize=3)
for n, label in [(8, '8비트\n256'), (16, '16비트\n65,536'), (32, '32비트\n43억')]:
    plt.annotate(label, (n, 2.0 ** n), textcoords='offset points', xytext=(6, -18), fontsize=8)
plt.xlabel('비트 수 n'); plt.ylabel('가짓수 2ⁿ (로그 눈금)'); plt.grid(alpha=0.3, which='both')
plt.title('스위치 하나가 늘 때마다 × 2 — n비트 칸의 천장'); plt.tight_layout(); plt.show()

assert 2.0 ** 32 > 4e9, '32비트면 약 43억 가지'
print('✔ 확인: 로그 눈금에서 직선 = 지수 성장 — 비트 몇 개 차이가 가짓수를 천문학적으로 바꾼다')

**직접 해보기**: ① `f'{나이:08b}'` 로 자기 나이를 이진수로 적어 보세요.
② 3절의 `bits` 배열에 200을 넣으면 int8 로는 몇으로 읽힐까요 — 예측한 뒤 확인해 보세요
(힌트: 200 = 128 + 64 + 8, 맨 앞 스위치가 켜져 있으니 음수).

## 5. 관찰 정리 + 🤖 AI 연결

- 이진법 = 자리값이 2배씩 자라는 셈법(1, 2, 4, 8, …) — 어떤 수든 자리값 몇 개의 합.
- **n비트 칸의 천장 = 2ⁿ 가지** (부호 없는 최대 2ⁿ − 1) — 토큰 번호 32,000개는 15비트.
- **2의 보수** = 맨 앞 자리값만 −2ⁿ⁻¹ 로 읽는 음수 약속 → int8 범위 **−128~127**,
  127 + 1 은 −128 로 감긴다(시계 산수). 비트는 그대로, 약속만 다르다.
- **회수하는 수학**: 5부 1~2장(스무고개의 비트 — 정보의 단위가 저장의 단위로),
  2부 7장(log₂ 로 필요한 비트 수 계산).
- **8장 예고**: int8 양자화 = 가중치 70억 개를 오늘 본 **256눈금 칸**에 옮겨 담는 기술.

**다음 챕터**: 3장 — 고무줄 자(부동소수점). 정수 자로는 0.1도, 10³⁸ 도 못 담습니다 —
지수를 도입해 눈금이 늘어나는 자를 만드는 법, fp32 를 해부합니다.